# SpaceX IPO forecast — a live calibration test

Claude's knowledge cutoff is **January 2026**; today is **August 12, 2026**.
As of the cutoff: December 2025 reporting said SpaceX was in *early internal
discussions* about a potential IPO (possibly Starlink-only), "as soon as late
2026", after tender offers around a $350-400B valuation. Nothing filed.

This notebook encodes Claude's mechanistic beliefs as string-DSL estimates,
fits the maxent flow sampler to them, and reads out the implied joint —
then a resolution cell scores the fitted marginals against what actually
happened (which Claude does not know).

Causal skeleton: drivers (`musk_intent`, `market_window`) and blockers
(`starship_setback`) feed a mechanical pipeline
`musk_intent -> board_decision -> s1_filed -> ipo_by_aug / ipo_by_eoy`,
with structure (`starlink_only`) and `valuation_b` conditioned on listing.

**Runtime -> GPU**, run top to bottom; edit the estimate cell freely.

In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

%pip -q install optax jaxopt pydantic

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
if jax.default_backend() == "cpu":
    print("WARNING: no GPU — the fit will run but compile is ~10x slower")
print("setup OK")

In [ ]:
# --- Variables: the causal skeleton -------------------------------------------
# Resolution criteria live in the descriptions; all dates are 2026.
from calibrated_response.models.variable import BinaryVariable, ContinuousVariable

VARS = [
    # drivers / blockers
    BinaryVariable(name="musk_intent", description=(
        "Musk/SpaceX seriously pursued a public listing during Jan-Aug 2026 "
        "(an active internal IPO process at any point, not just chatter)")),
    BinaryVariable(name="market_window", description=(
        "The IPO window was broadly open in H1 2026 "
        "(major tech IPOs pricing at or above range)")),
    BinaryVariable(name="starship_setback", description=(
        "A major Starship failure or grounding in H1 2026 that materially "
        "muddied the growth story")),
    # mechanical pipeline
    BinaryVariable(name="board_decision", description=(
        "SpaceX's board formally approved pursuing an IPO by Aug 12")),
    BinaryVariable(name="s1_filed", description=(
        "An S-1 or equivalent (incl. a confidential draft becoming public) "
        "was filed by Aug 12")),
    BinaryVariable(name="ipo_by_aug", description=(
        "SpaceX or Starlink shares were trading on a public exchange "
        "by Aug 12, 2026")),
    BinaryVariable(name="ipo_by_eoy", description=(
        "SpaceX or Starlink shares trading publicly by Dec 31, 2026")),
    # structure + size
    BinaryVariable(name="starlink_only", description=(
        "The listed (or to-be-listed) entity is Starlink spun out, "
        "rather than all of SpaceX")),
    ContinuousVariable(name="valuation_b", description=(
        "Valuation at IPO pricing, or the latest private tender mark if "
        "no IPO happened, in USD billions"),
        lower_bound=100.0, upper_bound=1200.0, unit="USD billions"),
]
print(f"{len(VARS)} variables:", ", ".join(v.name for v in VARS))

In [ ]:
# --- Claude's beliefs, in the string DSL ---------------------------------------
# `~ w` = self-stated uncertainty (log-odds for P: ~1.5 weak hunch, ~0.5
# confident; value units for E). All numbers were stated BEFORE resolution.
from calibrated_response.models.natural_response import parse_natural_syntax as est

ESTIMATES = [
    # drivers and blockers (base rates as of Jan 2026)
    est("P(musk_intent = True) = 0.45 ~ 0.8"),      # Dec-2025 talk was real but Musk walks things back
    est("P(market_window = True) = 0.70 ~ 1.0"),    # windows are open more often than not
    est("P(starship_setback = True) = 0.35 ~ 1.0"), # base rate of RUD/groundings per half-year

    # the mechanical pipeline: intent -> board -> filing -> listing
    est("P(board_decision = True | musk_intent = True) = 0.55 ~ 0.8"),
    est("P(board_decision = True | musk_intent = False) = 0.02 ~ 0.7"),
    est("P(s1_filed = True | board_decision = True) = 0.55 ~ 0.8"),
    est("P(s1_filed = True | board_decision = False) = 0.01 ~ 0.7"),
    # timeline math: S-1 -> pricing is 3-6 months, so an Aug listing needs a ~March filing
    est("P(ipo_by_aug = True | s1_filed = True) = 0.25 ~ 0.8"),
    est("P(ipo_by_aug = True | s1_filed = False) = 0.005 ~ 0.7"),
    est("P(ipo_by_eoy = True | s1_filed = True) = 0.75 ~ 0.8"),
    est("P(ipo_by_eoy = True | s1_filed = False, musk_intent = True) = 0.15 ~ 1.0"),
    est("P(ipo_by_eoy = True | musk_intent = False) = 0.01 ~ 0.7"),
    # implication: listed by August => listed by end of year
    est("P(ipo_by_eoy = True | ipo_by_aug = True) = 0.99 ~ 0.4"),

    # blockers gate completion
    est("P(ipo_by_eoy = True | market_window = False) = 0.04 ~ 0.9"),
    est("P(ipo_by_eoy = True | starship_setback = True) = 0.08 ~ 1.0"),

    # topline anchors (the directly-stated forecasts)
    est("P(ipo_by_aug = True) = 0.06 ~ 0.7"),
    est("P(ipo_by_eoy = True) = 0.22 ~ 0.8"),
    est("P(s1_filed = True) = 0.20 ~ 1.0"),

    # structure and size, conditional on a listing
    est("P(starlink_only = True | ipo_by_eoy = True) = 0.75 ~ 0.7"),
    est("E[valuation_b | ipo_by_eoy = True] = 550 ~ 150"),
    est("P(valuation_b > 400 | ipo_by_eoy = True) = 0.75 ~ 0.8"),
    est("P(valuation_b > 800 | ipo_by_eoy = True) = 0.15 ~ 0.9"),
    est("E[valuation_b | ipo_by_eoy = False] = 420 ~ 80"),  # the private mark drifts up
]

for e in ESTIMATES:
    sd = f"  sd={e.sd:g}" if getattr(e, "sd", None) is not None else ""
    print(f"{e.id[:40]:>40}  {e.to_query_estimate()}{sd}")

In [ ]:
# --- Fit the joint ---------------------------------------------------------------
from calibrated_response.maxent_sampler.distribution_builder import DistributionBuilder

STEPS, SEED = 2500, 0
builder = DistributionBuilder(VARS, ESTIMATES)
assert not builder.skipped, builder.skipped
if builder.warnings:
    print("warnings:", *builder.warnings, sep="\n  ")

builder.fit(steps=STEPS, lr=2e-3, n_samples=2048, seed=SEED)
s = builder.sample_dict(60_000, seed=SEED + 1)
print(f"fitted; {len(builder.constraints)} constraints")

In [ ]:
# --- Constraint report: does the fit honor the beliefs? --------------------------
rep = sorted(builder.constraint_report(), key=lambda c: -abs(c["error_rel"]))
print("worst-fitted constraints (top 10):")
for c in rep[:10]:
    print(f"{c['id'][:40]:>40}  target={c['target']:.3f} "
          f"fitted={c['fitted']:.3f}  err_rel={c['error_rel']:+.3f}")

In [ ]:
# --- Fitted marginals and key conditionals ---------------------------------------
import numpy as np

BINARIES = [v.name for v in VARS if v.name != "valuation_b"]
b = {n: (s[n] > 0.5) for n in BINARIES}
val = s["valuation_b"]

print("fitted marginals:")
for n in BINARIES:
    print(f"  P({n}) = {b[n].mean():.3f}")

def cond(a, given):
    return float(a[given].mean()) if given.sum() else float("nan")

print()
print(f"  P(ipo_by_eoy | s1_filed)      = {cond(b['ipo_by_eoy'], b['s1_filed']):.3f}")
print(f"  P(ipo_by_aug | ipo_by_eoy)    = {cond(b['ipo_by_aug'], b['ipo_by_eoy']):.3f}")
print(f"  P(starlink_only | ipo_by_eoy) = {cond(b['starlink_only'], b['ipo_by_eoy']):.3f}")
print(f"  E[valuation_b | ipo_by_eoy]   = {val[b['ipo_by_eoy']].mean():.0f}B")
print(f"  E[valuation_b | no ipo]       = {val[~b['ipo_by_eoy']].mean():.0f}B")

In [ ]:
# --- Pairwise plot over the pipeline + valuation -----------------------------------
from calibrated_response.maxent_sampler import plot_pairwise

NAMES = ["musk_intent", "s1_filed", "ipo_by_eoy", "valuation_b"]
sites = [builder.var_name_to_idx[n] for n in NAMES]
plot_pairwise(builder.model, builder.params, sites=sites, names=NAMES,
              n_samples=30_000, seed=11, bins=50);

In [ ]:
# --- RESOLUTION: fill in what actually happened, then run --------------------------
# Set each to True/False (leave None to skip); set the valuation if it resolved.
TRUTH = {
    "musk_intent":      None,
    "market_window":    None,
    "starship_setback": None,
    "board_decision":   None,
    "s1_filed":         None,
    "ipo_by_aug":       None,
    "ipo_by_eoy":       None,   # only scoreable after Dec 31 2026
    "starlink_only":    None,
}
TRUE_VALUATION_B = None   # e.g. 460.0

# Claude's directly-stated raw numbers (pre-solver), for the fused-vs-raw comparison
RAW = {"ipo_by_aug": 0.06, "ipo_by_eoy": 0.22, "s1_filed": 0.20,
       "musk_intent": 0.45, "market_window": 0.70, "starship_setback": 0.35}

import numpy as np
rows, brier_f, brier_r, log_f, log_r = [], [], [], [], []
for n, t in TRUTH.items():
    if t is None:
        continue
    p_fit = float(np.clip(b[n].mean(), 1e-4, 1 - 1e-4))
    y = float(t)
    lf = -(y * np.log(p_fit) + (1 - y) * np.log(1 - p_fit))
    brier_f.append((p_fit - y) ** 2); log_f.append(lf)
    line = f"{n:>18}  truth={t!s:>5}  fitted={p_fit:.3f}  log={lf:.3f}"
    if n in RAW:
        p_raw = np.clip(RAW[n], 1e-4, 1 - 1e-4)
        lr = -(y * np.log(p_raw) + (1 - y) * np.log(1 - p_raw))
        brier_r.append((p_raw - y) ** 2); log_r.append(lr)
        line += f"  |  raw={RAW[n]:.2f}  log={lr:.3f}"
    rows.append(line)

print(*rows, sep="\n")
if brier_f:
    print(f"\nfused : Brier={np.mean(brier_f):.4f}  mean log score={np.mean(log_f):.4f}")
if brier_r:
    print(f"raw   : Brier={np.mean(brier_r):.4f}  mean log score={np.mean(log_r):.4f}")
if TRUE_VALUATION_B is not None:
    z = (TRUE_VALUATION_B - val.mean()) / val.std()
    print(f"\nvaluation: truth {TRUE_VALUATION_B:.0f}B, "
          f"fitted {val.mean():.0f}B +- {val.std():.0f}B  (z={z:+.2f})")